<a href="https://colab.research.google.com/github/Hion-cy/ClassFiles/blob/main/Transformaciones_al263158.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

###**Materia:** Ingeniería de Datos Avanzada
###**Alumna:** Carmen Yolanda Hion Vela
###**Matrícula:** AL263158

Utilizando el archivo NSL_KDD.csv proporcionado en Moodle, desarrolle un notebook en Google Colab aplicando algunas de las transformaciones vistas en clase utilizando PySpark.

El archivo CSV deberá cargarse desde Google Drive. Para esta práctica puede utilizar carga automática del esquema mediante:

.option('inferSchema', 'true')
La práctica deberá incluir las siguientes actividades:

In [10]:
import pyspark
from pyspark.sql import SparkSession

In [14]:

try:
    import pyspark

    print("PySpark ya está instalado.")
    print(f"Versión instalada: {pyspark.__version__}")

except ImportError:

    print("PySpark no está instalado.")
    print("Instalando PySpark 3.5.0...\n")

    # Instalación silenciosa
    !pip install pyspark==3.5.0 -q

    # Verificación posterior a la instalación
    import pyspark

    print("PySpark instalado correctamente.")
    print(f"Versión instalada: {pyspark.__version__}")
    print("\nInformación desde terminal:")
!pyspark --version

PySpark ya está instalado.
Versión instalada: 4.0.2
Welcome to
      ____              __
     / __/__  ___ _____/ /__
    _\ \/ _ \/ _ `/ __/  '_/
   /___/ .__/\_,_/_/ /_/\_\   version 4.0.2
      /_/
                        
Using Scala version 2.13.16, OpenJDK 64-Bit Server VM, 17.0.18
Branch HEAD
Compiled by user runner on 2026-02-02T08:08:13Z
Revision 7cc3b9bcdaab8c923f23cdbc9ce922530e1becf1
Url https://github.com/apache/spark
Type --help for more information.


#1 Cargar el dataset en un DataFrame de PySpark.

In [75]:
from google.colab import drive
drive.mount('/content/drive')
ruta_del_archivo = '/content/drive/MyDrive/Classroom/NSL_KDD.csv'
#df=spark.read.csv(ruta_del_archivo, header=True, inferSchema=True, ignoreLeadingWhiteSpace=True)


#Renombrar cOLUMNAS
nombres_reales = [
     "protocol_type", "service", "flag", "src_bytes", "dst_bytes", "land",
    "wrong_fragment", "urgent", "hot", "num_failed_logins", "logged_in", "num_compromised",
    "root_shell", "su_attempted", "num_root", "num_file_creations", "num_shells",
    "num_access_files", "num_outbound_cmds", "is_host_login", "is_guest_login", "count",
    "srv_count", "serror_rate", "srv_serror_rate", "rerror_rate", "srv_rerror_rate",
    "same_srv_rate", "diff_srv_rate", "srv_diff_host_rate", "dst_host_count",
    "dst_host_srv_count", "dst_host_same_srv_rate", "dst_host_diff_srv_rate",
    "dst_host_same_src_port_rate", "dst_host_srv_diff_host_rate", "dst_host_serror_rate",
    "dst_host_srv_serror_rate", "dst_host_rerror_rate", "dst_host_srv_rerror_rate",
    "label", "difficulty_level"
]
# 2. Leer el archivo omitiendo cualquier cabecera rota (header=False)
# y asignando la lista completa de 43 elementos
df = spark.read.csv(ruta_del_archivo, header=True, inferSchema=True).toDF(*nombres_reales)

#Df = df.withColumnRenamed("class", "label")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


#2 Mostrar:
* Número total de filas.
* Número total de columnas.
* Primeras 10 filas utilizando show().

In [76]:
print("Número total de filas:", df.count())
print("Número total de columnas:", len(df.columns))
print('Primeras 10 filas')
print(df.show(10))

Número total de filas: 18035
Número total de columnas: 42
Primeras 10 filas
+-------------+-------+----+---------+---------+----+--------------+------+---+-----------------+---------+---------------+----------+------------+--------+------------------+----------+----------------+-----------------+-------------+--------------+-----+---------+-----------+---------------+-----------+---------------+-------------+-------------+------------------+--------------+------------------+----------------------+----------------------+---------------------------+---------------------------+--------------------+------------------------+--------------------+------------------------+------------+----------------+
|protocol_type|service|flag|src_bytes|dst_bytes|land|wrong_fragment|urgent|hot|num_failed_logins|logged_in|num_compromised|root_shell|su_attempted|num_root|num_file_creations|num_shells|num_access_files|num_outbound_cmds|is_host_login|is_guest_login|count|srv_count|serror_rate|srv_serror_rate|re

#3 Seleccionar únicamente las siguientes columnas utilizando select():
* protocol_type
* service
* src_bytes
* dst_bytes
* label

In [77]:
df_col_select = df.select("protocol_type", "service", "src_bytes", "dst_bytes", "label")
df_col_select.show(10)

+-------------+-------+---------+---------+------------+
|protocol_type|service|src_bytes|dst_bytes|       label|
+-------------+-------+---------+---------+------------+
|          tcp|private|        0|        0|     neptune|
|          tcp|private|        0|        0|     neptune|
|          tcp| telnet|        0|       15|       mscan|
|          tcp|   http|      267|    14515|      normal|
|          tcp| telnet|      129|      174|guess_passwd|
|          tcp|   http|      327|      467|      normal|
|          tcp|    ftp|       26|      157|guess_passwd|
|          tcp| telnet|        0|        0|       mscan|
|          tcp|private|        0|        0|     neptune|
|          tcp| telnet|        0|        0|     neptune|
+-------------+-------+---------+---------+------------+
only showing top 10 rows


##Comentarios adicionales

Las columnas del dataset se recorren debido a la existencia de 'duration' por lo que se reviso todo el csv y se corrigio el nombre de cada columna.

#4 Filtrar únicamente los registros donde:
* protocol_type = 'tcp'
y src_bytes > 1000

In [90]:
df_filtro = df_col_select.filter((df_col_select.protocol_type == 'tcp') & (df_col_select.src_bytes > 1000))
print('Número total de filas: ', df_filtro.count())
df_filtro.show(10)

Número total de filas:  1552
+-------------+--------+---------+---------+-----------+
|protocol_type| service|src_bytes|dst_bytes|      label|
+-------------+--------+---------+---------+-----------+
|          tcp|    http|    76944|        1|    apache2|
|          tcp|ftp_data|   283618|        0|warezmaster|
|          tcp|    http|    72564|        0|    apache2|
|          tcp|    smtp|     2599|      293|   mailbomb|
|          tcp|    smtp|     4030|      332|     normal|
|          tcp|    http|    72564|        0|    apache2|
|          tcp|    http|    54540|     8314|       back|
|          tcp|    smtp|     1376|      343|     normal|
|          tcp|    smtp|     1500|      332|     normal|
|          tcp|    smtp|     1710|      366|     normal|
+-------------+--------+---------+---------+-----------+
only showing top 10 rows


##Comentarios adicionales
Al aplicar la condicion el volumen del conjunto de datos se redujo de 18035 a 1552 filas.

#5 Crear una nueva columna
Llamada:

total_bytes
La nueva columna deberá contener:

src_bytes + dst_bytes


In [82]:
df_filtro=df_filtro.withColumn("total_bytes", df_filtro.src_bytes + df_filtro.dst_bytes)
df_filtro.show(10)

+-------------+--------+---------+---------+-----------+-----------+
|protocol_type| service|src_bytes|dst_bytes|      label|total_bytes|
+-------------+--------+---------+---------+-----------+-----------+
|          tcp|    http|    76944|        1|    apache2|      76945|
|          tcp|ftp_data|   283618|        0|warezmaster|     283618|
|          tcp|    http|    72564|        0|    apache2|      72564|
|          tcp|    smtp|     2599|      293|   mailbomb|       2892|
|          tcp|    smtp|     4030|      332|     normal|       4362|
|          tcp|    http|    72564|        0|    apache2|      72564|
|          tcp|    http|    54540|     8314|       back|      62854|
|          tcp|    smtp|     1376|      343|     normal|       1719|
|          tcp|    smtp|     1500|      332|     normal|       1832|
|          tcp|    smtp|     1710|      366|     normal|       2076|
+-------------+--------+---------+---------+-----------+-----------+
only showing top 10 rows


#6 Ordenar los registros
De mayor a menor utilizando la columna:
total_bytes

In [83]:
df_filtro=df_filtro.orderBy("total_bytes", ascending=False)
df_filtro.show(10)

+-------------+--------+---------+---------+------+-----------+
|protocol_type| service|src_bytes|dst_bytes| label|total_bytes|
+-------------+--------+---------+---------+------+-----------+
|          tcp|     X11| 62825648|    90476| xlock|   62916124|
|          tcp|     X11| 31645608|   207796| xlock|   31853404|
|          tcp|ftp_data|  6291668|        0|normal|    6291668|
|          tcp|ftp_data|  3131464|        0|normal|    3131464|
|          tcp|ftp_data|  2194619|        0|normal|    2194619|
|          tcp|     X11|    13948|  1171108|normal|    1185056|
|          tcp|     X11|   286040|   383476|normal|     669516|
|          tcp|     X11|    39224|   511712| named|     550936|
|          tcp|ftp_data|   501760|        0|normal|     501760|
|          tcp|ftp_data|   501760|        0|normal|     501760|
+-------------+--------+---------+---------+------+-----------+
only showing top 10 rows


##Comentarios adicionales
Al ordenar el conjunto de datos, se observa que los registros con mayor volumen de trafico para el protocolo tcp corresponden a los servicios X11 y ftp_data.

#7 Mostrar los valores únicos de: protocol_type

In [85]:
df_protocol_unique=df_col_select.select("protocol_type").distinct()
df_protocol_unique.show()

+-------------+
|protocol_type|
+-------------+
|          tcp|
|          udp|
|         icmp|
+-------------+



##Comentarios adicionales
Existen 3 tipos de protocolos en el conjunto de datos: tcp, udp, icmp

#8 Agrupar los registros
Utilizando:

groupBy('protocol_type')
y calcular:

el número de registros utilizando count().

In [88]:
df_group_protocol=df_col_select.groupBy('protocol_type').count()
df_group_protocol.show()

+-------------+-----+
|protocol_type|count|
+-------------+-----+
|          tcp|15136|
|          udp| 2074|
|         icmp|  825|
+-------------+-----+



##Comentarios adicionales
El protocolo que cuanta con mas registros es tcp

#9 Agrupar
Por:

* label

y calcular:

* el promedio de src_bytes.

In [89]:
df_group_label=df_col_select.groupBy('label').avg('src_bytes')
df_group_label.show()


+---------------+------------------+
|          label|    avg(src_bytes)|
+---------------+------------------+
|             ps|124.58333333333333|
|        neptune|               0.0|
|          satan|0.5208333333333334|
|          saint| 6.637065637065637|
|           nmap|               0.0|
|        apache2| 38689.75465313029|
|      portsweep|               0.0|
|           back| 52894.35051546392|
|      sqlattack|             398.0|
|         xsnoop|            775.75|
|   guess_passwd| 57.58841778697001|
|         normal| 2501.925664398511|
|        rootkit|           54713.2|
|           perl|             258.0|
|     httptunnel|506.83653846153845|
|buffer_overflow|            1867.0|
|       udpstorm|               0.0|
|       multihop|1767.8333333333333|
|        ipsweep|14.666666666666666|
|           worm|            4209.0|
+---------------+------------------+
only showing top 20 rows


#Comentarios finales
En esta práctica se aplicaron las transformaciones de PySpark de:
* Select
* Filter
* withColumn
* orderBy
* groupBy
Sobre el dataset de ciberseguridad de NSL-KDD, entre los hallazgos principales se tuvo que realizar una alineacion de los datos ya que existia una columna de 'duration en el CSV que desplazaba los nombres de las columnas 1 hacia la derecha.

Ademas la columna 'label' no existia como tal por lo que tuvo que renombrarse

El análisis final demostró que ataques como back y apache2 saturan la red enviando paquetes muy pesados llenos de datos, mientras que alertas como neptune no envían datos (0.0 bytes) porque solo hacen pruebas o escaneos rápidos para saturar la conexión.